In [5]:
import numpy as np
from Bio import SeqIO, motifs
from Bio.Seq import Seq

CHR1_PATH  = "chr1.fa"
REGION_LEN = 1_000_000
THRESHOLD  = 5.0
NUCLEOTIDES = ["A", "C", "G", "T"]
BG = {"A": 0.295, "C": 0.205, "G": 0.205, "T": 0.295}
BG_PROBS = np.array([BG[n] for n in NUCLEOTIDES])
sites_raw = ["GAGGTAAAC", "TCCGTAAGC", "CAGGTTGGA", "ACAGTCAGC", "TAGGTCAGC", "CAGGTCAGC", "CAGGTCGAT", "CAGGTCAGC", "CAGGTCAGC", "CAGGTTGGC",]
instances = [Seq(s) for s in sites_raw]
m = motifs.create(instances)
m.background   = BG
m.pseudocounts = 0.1
print(f"Консенсус: {m.consensus}")
L = len(sites_raw[0])
pwm_np = np.array([[m.pwm[nuc][i] for i in range(L)] for nuc in NUCLEOTIDES])          

NUC2IDX = {n: i for i, n in enumerate(NUCLEOTIDES)}
NUC2IDX["N"] = -1
def score_sequence_fast(seq_str: str, pwm: np.ndarray) -> np.ndarray:

    motif_len = pwm.shape[1]
    encoded   = np.array([NUC2IDX.get(c, -1) for c in seq_str], dtype=np.int8)
    n_windows = len(encoded) - motif_len + 1

    idx_matrix = np.lib.stride_tricks.sliding_window_view(encoded, motif_len)
    valid   = (idx_matrix >= 0).all(axis=1)
    col_idx = np.arange(motif_len)

    scores = np.full(n_windows, -np.inf)
    valid_windows  = idx_matrix[valid]
    scores[valid]  = pwm[valid_windows, col_idx].sum(axis=1)
    return scores

record   = next(SeqIO.parse(CHR1_PATH, "fasta"))
sequence = str(record.seq[:REGION_LEN]).upper()
scores_fwd = score_sequence_fast(sequence, pwm_np)
hits_fwd   = [(int(p), "+", round(float(scores_fwd[p]), 4)) for p in np.where(scores_fwd > THRESHOLD)[0]]
print(f"Хитов на прямой цепи: {len(hits_fwd)}")

rev_seq    = str(Seq(sequence).reverse_complement())
scores_rev = score_sequence_fast(rev_seq, pwm_np)
hits_rev   = [(len(sequence) - int(p) - L, "-", round(float(scores_rev[p]), 4)) for p in np.where(scores_rev > THRESHOLD)[0]]
print(f"Хитов на обратной цепи: {len(hits_rev)}")

all_hits = sorted(hits_fwd + hits_rev, key=lambda x: x[0])

print(f"Найдено хитов: {len(all_hits)}  (скор > {THRESHOLD})")
print(f"Прямая (+): {len(hits_fwd)}")
print(f"Обратная (-): {len(hits_rev)}")
print(f"  {'Позиция':>10}  {'Цепь':^6}  {'Скор':>10}")
for pos, strand, score in all_hits:
    print(f"  {pos:>10,}  {strand:^6}  {score:>10.4f}")

print("\nТоп-5 хитов:")
top5 = sorted(all_hits, key=lambda x: -x[2])[:5]
for pos, strand, score in top5:
    if strand == "+":
        seq_hit = sequence[pos:pos + L]
    else:
        rpos = len(sequence) - pos - L
        seq_hit = rev_seq[rpos:rpos + L]
    print(f"  pos={pos:>9,}  {strand}  score={score:.4f}  {seq_hit}")

Консенсус: CAGGTCAGC
Хитов на прямой цепи: 2896
Хитов на обратной цепи: 2717
Найдено хитов: 5613  (скор > 5.0)
Прямая (+): 2896
Обратная (-): 2717
     Позиция   Цепь         Скор
      10,172    -         5.3750
      10,500    -         6.2404
      10,564    +         5.2788
      11,404    -         5.9519
      11,868    -         5.2788
      11,876    +         5.1827
      11,891    -         5.1827
      12,012    -         5.0865
      12,029    -         5.9519
      12,294    -         5.3750
      12,375    +         5.4712
      12,413    +         5.1827
      12,417    +         5.1827
      12,579    -         5.3750
      12,648    +         5.4712
      12,676    -         5.8558
      12,718    +         5.1827
      12,971    +         5.2788
      12,983    +         5.1827
      13,015    -         5.0865
      13,049    +         5.1827
      13,123    -         5.1827
      13,134    -         5.4712
      13,288    -         5.1827
      13,305    -         5.